# Chapter 8: Using Convolutions to Generalize
### Convolutions, Handcrafted Filters, Pooling, Modular Subclassing, Regularization (L2, Dropout, BatchNorm), and Deep ResNets

This companion notebook implements the complete pedagogical workflow of **Chapter 8** from *Deep Learning with PyTorch (2nd Edition)* by Eli Stevens, Luca Antiga, Thomas Viehmann, and Howard Huang.

#### Contents:
1. Setup & Device Configuration
2. CIFAR-2 Dataset Preparation (Birds vs Airplanes)
3. Convolutions in Action: Handcrafted Feature Detectors (Edge, Horizontal, Vertical, Blur)
4. Boundary Conditions & Padding Mechanics
5. Spatial Downsampling: 2x2 Max Pooling & Receptive Field Expansion
6. Building the Baseline CNN (`Net`) with `nn.Module`
7. Parameter Count & Memory Efficiency Analysis
8. Clean Architecture: Refactoring with `torch.nn.functional`
9. Mini-Batch Training Loop and Validation Pipeline
10. Model Capacity Exploration: Width vs Depth
11. Regularization Strategies: L2 Weight Decay & Spatial Dropout
12. Stabilizing Internal Covariate Shift: Batch Normalization (`nn.BatchNorm2d`)
13. Going Deeper: Residual Connections (`NetRes`) and Ultra-Deep ResNets (`NetResDeep`)
14. Empirical Architecture Benchmarking & Results Visualization

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set deterministic seeds
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch Version:     {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Active Device:       {device}")

## 1. CIFAR-2 Dataset Preparation (Birds vs Airplanes)
Following Chapter 7, we normalize the images channel-wise by subtracting the dataset means `(0.4914, 0.4822, 0.4465)` and dividing by standard deviations `(0.2470, 0.2435, 0.2616)`. We then filter the dataset to class 0 (`airplane`) and class 2 (`bird`), re-indexing class 2 to target index 1.

In [ ]:
data_path = Path('./data')
data_path.mkdir(parents=True, exist_ok=True)

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar10_train = datasets.CIFAR10(root=str(data_path), train=True, download=True, transform=cifar_transform)
cifar10_val = datasets.CIFAR10(root=str(data_path), train=False, download=True, transform=cifar_transform)

# Binary subset: airplane (0) and bird (2)
label_map = {0: 0, 2: 1}
class_names = ['airplane', 'bird']

cifar2_train = [(img, label_map[label]) for img, label in cifar10_train if label in [0, 2]]
cifar2_val = [(img, label_map[label]) for img, label in cifar10_val if label in [0, 2]]

print(f"Training samples:   {len(cifar2_train)}")
print(f"Validation samples: {len(cifar2_val)}")

## 2. Convolutions in Action: Handcrafted Feature Detectors
Before training kernels via gradient descent, we explore how 2D convolutions function by manually setting the kernel weights to detect edges, horizontal bands, vertical lines, and smoothing blur on a grayscale sample.

In [ ]:
# Extract a sample image and convert to single-channel luminance
sample_img, _ = cifar2_train[0]
sample_gray = sample_img.mean(dim=0, keepdim=True).unsqueeze(0)  # Shape: (1, 1, 32, 32)

# Define handcrafted 3x3 kernels
kernels = {
    'CONV_EDGE': torch.tensor([
        [-1.0, -1.0, -1.0],
        [-1.0,  8.0, -1.0],
        [-1.0, -1.0, -1.0]
    ]),
    'CONV_HORIZONTAL': torch.tensor([
        [-1.0, -2.0, -1.0],
        [ 0.0,  0.0,  0.0],
        [ 1.0,  2.0,  1.0]
    ]),
    'CONV_VERTICAL': torch.tensor([
        [-1.0, 0.0, 1.0],
        [-2.0, 0.0, 2.0],
        [-1.0, 0.0, 1.0]
    ]),
    'CONV_BLUR': torch.tensor([
        [1.0/9, 1.0/9, 1.0/9],
        [1.0/9, 1.0/9, 1.0/9],
        [1.0/9, 1.0/9, 1.0/9]
    ])
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
axes[0].imshow(sample_gray.squeeze().numpy(), cmap='gray')
axes[0].set_title('INPUT')
axes[0].axis('off')

with torch.no_grad():
    for i, (name, k) in enumerate(kernels.items(), start=1):
        conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1, bias=False)
        conv.weight.data = k.unsqueeze(0).unsqueeze(0)
        filtered = conv(sample_gray)
        axes[i].imshow(filtered.squeeze().numpy(), cmap='gray')
        axes[i].set_title(name)
        axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Boundary Handling & Spatial Downsampling
By default, a $3 \times 3$ kernel without padding shrinks an $H \times W$ image to $(H-2) \times (W-2)$. Adding `padding=1` preserves the spatial resolution. We then apply `nn.MaxPool2d(2)` to subsample the image by half along each spatial dimension.

In [ ]:
# Verify shape preservation with padding
conv_padded = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
sample_batch = sample_img.unsqueeze(0)
out_padded = conv_padded(sample_batch)
print(f"Input shape:               {sample_batch.shape}")
print(f"Conv2d (padding=1) shape:  {out_padded.shape}")

# Apply 2x2 Max Pooling
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_pooled = pool(out_padded)
print(f"MaxPool2d (2x2) shape:     {out_pooled.shape}")

## 4. Building the Baseline CNN (`Net`)
We assemble the baseline convolutional network following Chapter 8:
- Layer 1: `Conv2d(3, 16, kernel_size=3, padding=1)` $\to$ `Tanh` $\to$ `MaxPool2d(2)` $(16 \times 16 \times 16)$
- Layer 2: `Conv2d(16, 8, kernel_size=3, padding=1)` $\to$ `Tanh` $\to$ `MaxPool2d(2)` $(8 \times 8 \times 8)$
- Flattening: `view(-1, 8 * 8 * 8)` $= 512$
- Head: `Linear(512, 32)` $\to$ `Tanh` $\to$ `Linear(32, 2)`

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.act1 = nn.Tanh()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.act2 = nn.Tanh()
        self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(8 * 8 * 8, 32)
        self.act3 = nn.Tanh()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = self.pool1(self.act1(self.conv1(x)))
        out = self.pool2(self.act2(self.conv2(out)))
        out = out.view(-1, 8 * 8 * 8)
        out = self.act3(self.fc1(out))
        out = self.fc2(out)
        return out

model = Net().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters in Baseline Net: {total_params:,}")

## 5. Refactoring with `torch.nn.functional`
Stateless operations (activations, pooling) do not possess learnable parameters. We refactor `Net` using functional calls `F.tanh` (or `F.relu`) and `F.max_pool2d`.

In [ ]:
class NetFunctional(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        out = out.view(-1, 8 * 8 * 8)
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

net_func = NetFunctional().to(device)
dummy_input = torch.randn(4, 3, 32, 32, device=device)
print(f"Output logit shape: {net_func(dummy_input).shape}")

## 6. Training & Validation Pipeline
We encapsulate training and validation into a reusable function, measuring epoch loss and computing binary classification accuracy across both partitions.

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2_train, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64, shuffle=False)

def train_model(model, train_loader, val_loader, optimizer, criterion, epochs=15, device=device):
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
            
        # Validation pass
        model.eval()
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, preds = torch.max(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
                
        epoch_loss = total_loss / total_train
        train_acc = correct_train / total_train
        val_acc = correct_val / total_val
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:2d}/{epochs:2d} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
            
    return history

# Train baseline model
baseline_net = NetFunctional().to(device)
optimizer = optim.SGD(baseline_net.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

print("Training Baseline ConvNet:")
baseline_history = train_model(baseline_net, train_loader, val_loader, optimizer, criterion, epochs=15)

## 7. Controlling Capacity & Regularization
We investigate four techniques to balance model capacity and fight overfitting:
1. **Width:** Increasing channels ($32 \to 16$ instead of $16 \to 8$).
2. **L2 Regularization (Weight Decay):** Adding penalty $\lambda \sum w^2$ directly via optimizer `weight_decay`.
3. **Dropout:** Stochastically zeroing channels with `nn.Dropout2d`.
4. **Batch Normalization:** Normalizing feature activations across the batch with `nn.BatchNorm2d`.

In [ ]:
# Model with increased width
class NetWidth(nn.Module):
    def __init__(self, n_chans=32):
        super().__init__()
        self.n_chans = n_chans
        self.conv1 = nn.Conv2d(3, n_chans, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(n_chans, n_chans // 2, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8 * 8 * (n_chans // 2), 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        out = out.view(-1, 8 * 8 * (self.n_chans // 2))
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

# Model with Dropout
class NetDropout(nn.Module):
    def __init__(self, n_chans=32):
        super().__init__()
        self.n_chans = n_chans
        self.conv1 = nn.Conv2d(3, n_chans, kernel_size=3, padding=1)
        self.conv1_dropout = nn.Dropout2d(p=0.4)
        self.conv2 = nn.Conv2d(n_chans, n_chans // 2, kernel_size=3, padding=1)
        self.conv2_dropout = nn.Dropout2d(p=0.4)
        self.fc1 = nn.Linear(8 * 8 * (n_chans // 2), 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(self.conv1_dropout(torch.tanh(self.conv1(x))), 2)
        out = F.max_pool2d(self.conv2_dropout(torch.tanh(self.conv2(out))), 2)
        out = out.view(-1, 8 * 8 * (self.n_chans // 2))
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

# Model with Batch Normalization
class NetBatchNorm(nn.Module):
    def __init__(self, n_chans=32):
        super().__init__()
        self.n_chans = n_chans
        self.conv1 = nn.Conv2d(3, n_chans, kernel_size=3, padding=1)
        self.conv1_bn = nn.BatchNorm2d(num_features=n_chans)
        self.conv2 = nn.Conv2d(n_chans, n_chans // 2, kernel_size=3, padding=1)
        self.conv2_bn = nn.BatchNorm2d(num_features=n_chans // 2)
        self.fc1 = nn.Linear(8 * 8 * (n_chans // 2), 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = self.conv1_bn(self.conv1(x))
        out = F.max_pool2d(torch.relu(out), 2)
        out = self.conv2_bn(self.conv2(out))
        out = F.max_pool2d(torch.relu(out), 2)
        out = out.view(-1, 8 * 8 * (self.n_chans // 2))
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out

print("Regularized architectures instantiated successfully.")

## 8. Going Deeper: Residual Networks (`NetRes` & `NetResDeep`)
When stacking many convolutional layers, gradients vanish during backpropagation. He et al. (2015) introduced skip connections: $\mathcal{H}(x) = \mathcal{F}(x) + x$. This identity shortcut allows gradient signals to flow directly through hundreds of layers without attenuation.

In [ ]:
# Residual block with identity skip connection
class ResBlock(nn.Module):
    def __init__(self, n_chans):
        super().__init__()
        self.conv = nn.Conv2d(n_chans, n_chans, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(num_features=n_chans)
        
    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = torch.relu(out)
        return out + x  # Residual addition

# Deep ResNet stacking 100 residual blocks
class NetResDeep(nn.Module):
    def __init__(self, n_chans=32, n_blocks=100):
        super().__init__()
        self.n_chans = n_chans
        self.conv1 = nn.Conv2d(3, n_chans, kernel_size=3, padding=1)
        self.resblocks = nn.Sequential(
            *(n_blocks * [ResBlock(n_chans=n_chans)])
        )
        self.fc1 = nn.Linear(8 * 8 * n_chans, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.relu(self.conv1(x)), 2)
        out = self.resblocks(out)
        out = F.max_pool2d(out, 2)
        out = out.view(-1, 8 * 8 * self.n_chans)
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out

net_res_deep = NetResDeep(n_chans=32, n_blocks=100).to(device)
deep_params = sum(p.numel() for p in net_res_deep.parameters() if p.requires_grad)
print(f"NetResDeep (100 blocks) parameters: {deep_params:,}")

## 9. Empirical Benchmark Comparison
We visualize the empirical accuracy results across all tested architectural variants (reproducing Figure 8.12 from the textbook) to observe the trade-offs between width, depth, regularization, and skip connections.

In [ ]:
models_benchmark = ['BASELINE', 'WIDTH', 'L2 REG', 'DROPOUT', 'BATCH_NORM', 'DEPTH', 'RES', 'RES DEEP']
train_accuracies = [0.938, 0.968, 0.908, 0.902, 0.998, 0.958, 0.971, 0.976]
val_accuracies   = [0.897, 0.904, 0.879, 0.885, 0.899, 0.910, 0.903, 0.872]

x = np.arange(len(models_benchmark))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, train_accuracies, width, label='TRAIN', color='#4A90E2', edgecolor='#222')
rects2 = ax.bar(x + width/2, val_accuracies, width, label='VAL', color='#E67E22', edgecolor='#222')

ax.set_ylabel('ACCURACY', fontsize=12, fontweight='bold')
ax.set_title('Empirical Accuracy Benchmark: Train vs Validation Across Architectures', fontsize=13, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models_benchmark, rotation=45, ha='right', fontsize=11)
ax.set_ylim(0.70, 1.02)
ax.legend(loc='lower right', frameon=True, fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()